In [1]:
# Librerias

import pandas as pd
from pathlib import Path
import shutil
import ast #Para convertir strings en listas

In [3]:
# Funciones

# Funcion para construir la ruta donde se encuentran las imagenes

def construir_ruta(row):
    p = str(row["Path"]).strip() if pd.notna(row["Path"]) else ""
    ip = str(row["Image_Path"]).strip() if pd.notna(row["Image_Path"]) else ""
    name = str(row["Image_Name"]).strip() if pd.notna(row["Image_Name"]) else ""
    return Path(p) / ip / name

# Funcion para forzar que todos los id sean iguales a uno

def forzar_class_id(lista):
    nueva = []
    for item in lista:
        partes = item.strip().split()   # separa en tokens
        if len(partes) >= 5:
            partes[0] = "1"             # reemplaza class_id por 1
            nueva.append(" ".join(partes[:5]))
        elif len(partes) == 4:          # si no trae class_id, lo agregamos
            nueva.append("1 " + " ".join(partes))
    return nueva

In [4]:
# 1) Leer el CSV

CSV_PATH = "..\\..\\CSVPruebas\\Definitivos\\DB_Embarcaciones.csv"

df = pd.read_csv(CSV_PATH, delimiter=",")

# 2) Filtrar solo Main_Label == 1 (funciona si la columna es int o string)
df_positivos = df[df["Main_Label"].astype(str).str.strip() == "1"].copy()

# 3) Construir la ruta donde se encuentran las imagenes

df_positivos["ruta_imagen"] = df_positivos.apply(construir_ruta, axis=1)

Generar la carpeta de imagenes

In [7]:
# Carpeta de salida
out_dir = Path("../../DatasetAMZ/images")
out_dir.mkdir(parents=True, exist_ok=True)

# Recorremos cada fila con ruta_imagen
copiadas, faltantes = 0, 0
for ruta in df_positivos["ruta_imagen"]:
    src = Path("..") / Path(ruta)        # ubicación original
    dst = out_dir / src.name  # misma imagen pero en DatasetAMZ/images
    
    if src.exists():  # si la imagen realmente existe en disco
        shutil.copy2(src, dst)  # copia preservando metadata (fecha, etc.)
        copiadas += 1
    else:
        faltantes += 1

print(f"Copiadas: {copiadas}, No encontradas: {faltantes}")

Copiadas: 1877, No encontradas: 0


Generar la carpeta de labels

In [8]:
# Carpeta donde guardarás los .txt
out_dir = Path("../../DatasetAMZ/labels")
out_dir.mkdir(parents=True, exist_ok=True)

for idx, row in df_positivos.iterrows():
    # 1. nombre del archivo (cambiar .jpg → .txt)
    name = str(row["Image_Name"]).replace(".jpg", ".txt")
    out_file = out_dir / name

    # 2. convertir Segment (string → lista)
    try:
        texto = ast.literal_eval(row["Segment"])
    except Exception:
        # si Segment no está en formato lista, lo dejamos como lista de un elemento
        texto = [row["Segment"]]

    # 3. forzar class_id = 1
    texto = forzar_class_id(texto)

    # 4. guardar en archivo .txt
    with open(out_file, "w", encoding="utf-8") as f:
        for linea in texto:
            f.write(linea.strip() + "\n")